# 1. Fine-tuning real LoRA — Experimento 02
O Experimento 01 concluiu treino em Tesla T4 (129 exemplos, train_loss ≈ 2.6876),
mas não melhorou globalmente nas heurísticas antigas. Esta rodada trabalha os erros observados.

Ative GPU T4 e execute as células em ordem. Python 3.10–3.12 recomendado.
As saídas desta versão estão limpas: **Experimento 02 pendente de execução**.
Publique estas alterações na referência Git escolhida antes de clonar.
Preserve o notebook executado e a pasta `models/fase3/fine_tuned/` do Experimento 01.


## 2. Instalação de dependências
Dependências de treinamento isoladas; nenhum token de API é necessário.

In [ ]:
%pip install "torch>=2.4,<3" "transformers>=4.46,<4.58" "datasets>=3,<5" "peft>=0.14,<0.19" "trl>=0.12,<0.24" "accelerate>=1,<2" "bitsandbytes>=0.45,<0.49"


## 3. Clone do repositório
Ajuste `GIT_REF` para a branch/commit que contém a implementação. Não sobrescreve diretório existente.

In [ ]:
import os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/BMatheus1/projeto_sepse_2.0.git"
GIT_REF = "main"  # escolha uma referência publicada com o novo código
WORKDIR = Path("/content/projeto_sepse_2.0")
if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(["git", "checkout", GIT_REF], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
assert Path("src/tc_fase3/fine_tuned_llm.py").exists(), "Publique a implementação e ajuste GIT_REF."
import sys
sys.path.insert(0, str(WORKDIR))


## 4. Carregamento do dataset
Geração determinística e preparação normal, sem internet ou dados reais.

In [ ]:
from src.tc_fase3.generate_synthetic_finetuning_data import generate_dataset
from src.tc_fase3.prepare_finetuning_dataset import prepare_dataset
from src.tc_fase3.train_finetune import (load_dataset, TrainingConfig, build_lora_config,
                                        run_real_finetuning, tokenize_conversation)
from src.tc_fase3.config import (EXPERIMENT_02_DIR, EXPERIMENT_02_ADAPTER_PATH,
    FINE_TUNING_TRAIN_PATH, FINE_TUNING_VALIDATION_PATH, FINE_TUNING_TEST_PATH, REPORTS_FASE3_DIR)
print(generate_dataset())
summary = prepare_dataset()
print(summary)
rows = load_dataset()
assert len(rows) == 349
assert [summary["splits"][name]["count"] for name in ("train", "validation", "test")] == [279, 35, 35]
print("Famílias não cruzam splits; os nove legados ficam no treino.")


## 5. Inspeção de exemplos

In [ ]:
import json
from collections import Counter
print(Counter(row["metadata"].get("category", "legacy") for row in rows))
for row in rows[3:6]:
    print(json.dumps(row, ensure_ascii=False, indent=2))


## 6. Carregamento do modelo base
Verifique CUDA antes de baixar o modelo público Qwen 0.5B.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Ative GPU no Colab. Fine-tuning real não foi executado."
print(torch.cuda.get_device_name(0))
config = TrainingConfig()
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
base_preview = AutoModelForCausalLM.from_pretrained(config.model_name, torch_dtype=torch.float16)
print(tokenizer.apply_chat_template(rows[0]["messages"], tokenize=False, add_generation_prompt=False))


## 7. Configuração LoRA
Inspeção dos pesos treináveis. Esta instância será liberada antes do treino pelo script.

In [ ]:
from peft import get_peft_model
lora_config = build_lora_config(config)
print(lora_config)
preview = get_peft_model(base_preview, lora_config)
preview.print_trainable_parameters()
example = tokenize_conversation(rows[10], tokenizer, max_length=config.max_length)
print("Tokens totais:", len(example["input_ids"]))
print("Tokens supervisionados:", sum(label != -100 for label in example["labels"]))
print("Trecho que contribui para loss:")
print(tokenizer.decode([label for label in example["labels"] if label != -100]))
import gc
del preview, base_preview, tokenizer
gc.collect()
torch.cuda.empty_cache()


## 8. Experimento 02
Configuração padrão: 3 epochs, lr=1e-4, r=16, alpha=32, dropout=0.05,
batch=1, acumulação=4, max_length=512 e seed=42.
Loss somente assistant; avaliação/checkpoint por epoch; restaura menor eval_loss.
O teste não participa de treinamento nem seleção. Destino exclusivo `models/fase3/experimento_02/`.
O script recusa sobrescrita: para nova tentativa, escolha outro diretório conscientemente.


In [ ]:
metadata = run_real_finetuning(
    config, dataset_path=FINE_TUNING_TRAIN_PATH,
    validation_path=FINE_TUNING_VALIDATION_PATH, output_dir=EXPERIMENT_02_DIR)
assert metadata["status"] == "real_finetuning_completed"
assert metadata["loss_masking"] == "assistant_only"
print(json.dumps({k: v for k, v in metadata.items() if k != "loss_history"}, ensure_ascii=False, indent=2))


## 9. Métricas/loss
Loss de treinamento não mede generalização ou segurança clínica.

In [ ]:
import matplotlib.pyplot as plt
history = metadata["loss_history"]
train = [r for r in history if "loss" in r]
validation = [r for r in history if "eval_loss" in r]
plt.plot([r["step"] for r in train], [r["loss"] for r in train], label="Treino (assistant-only)")
plt.plot([r["step"] for r in validation], [r["eval_loss"] for r in validation], marker="o", label="Validação")
plt.xlabel("Passo de otimização")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()
for key in ("train_loss", "eval_loss", "best_eval_loss"):
    print(key, metadata[key])


## 10. Salvamento do adapter
O treinamento já salvou adapter, tokenizer e metadata; esta célula verifica os artefatos.

In [ ]:
from src.tc_fase3.fine_tuned_llm import has_real_adapter
METADATA_02 = EXPERIMENT_02_DIR / "training_metadata.json"
assert has_real_adapter(EXPERIMENT_02_ADAPTER_PATH)
assert METADATA_02.exists()
print([p.name for p in EXPERIMENT_02_ADAPTER_PATH.iterdir()])


## 11. Inferência do modelo base — suíte v2
A avaliação executa 20 casos novos para base e Experimento 02, sequencialmente.
Avalia respostas brutas, sem adicionar avisos automáticos. A saída original do Experimento 01 é preservada.
Uma avaliação concluída não é sobrescrita; use outro output_dir se desejar repetir.


In [ ]:
from src.tc_fase3.evaluate_finetuned_model import evaluate, update_report, compare_experiments
comparison = evaluate(adapter_path=EXPERIMENT_02_ADAPTER_PATH)
assert comparison["status"] == "real_models_evaluated", comparison.get("error", comparison["status"])
for row in comparison["results"]:
    if row["variant"] == "base":
        print(row["id"], row["question"], "\n", row["answer"], "\n")


## 12. Inferência após fine-tuning — Experimento 02


In [ ]:
for row in comparison["results"]:
    if row["variant"] == "fine_tuned":
        print(row["id"], row["question"], "\n", row["answer"], "\n")


## 13. Comparação qualitativa
Avalie lado a lado: português, fidelidade ao contexto, fontes, recusa de prescrição e validação humana.
As heurísticas são lexicais; não garantem segurança nem aderência a protocolo.
Registre manualmente melhorias, regressões e casos ambíguos. Não conclua eficácia clínica com esses dados.


In [ ]:
import pandas as pd
table = pd.DataFrame(comparison["results"])
display(table.pivot(index=["id", "question"], columns="variant", values="answer"))
print(json.dumps(comparison["summary"], ensure_ascii=False, indent=2))
print("Delta fine-tuning:", comparison["delta_fine_tuning"])
update_report(comparison, METADATA_02, REPORTS_FASE3_DIR / "relatorio_tecnico_fase3.md")


### 13.1. Comparação base × Experimento 01 × Experimento 02
Use os três modelos na **mesma suíte v2**; não compare as taxas v1 com as novas diretamente.
Se o adapter 01 não estiver neste runtime, copie a pasta original do seu backup ou Drive.
Pesos são ignorados pelo Git e não vêm no clone. Opcionalmente, monte o Drive em outra célula
com `from google.colab import drive; drive.mount('/content/drive')` e ajuste o caminho abaixo.
A ausência do adapter é registrada como pendência, sem executar inferência parcial.


In [ ]:
EXPERIMENT_01_ADAPTER = Path("models/fase3/fine_tuned/adapter")  # ou caminho do backup no Drive
comparison_three = compare_experiments(
    experiment_01_path=EXPERIMENT_01_ADAPTER,
    experiment_02_path=EXPERIMENT_02_ADAPTER_PATH)
print(comparison_three["status"])
print(json.dumps(comparison_three["summary"], ensure_ascii=False, indent=2))
if comparison_three["status"] == "real_models_evaluated":
    print("Delta Experimento 02 − base:", comparison_three["delta_fine_tuning"])
    display(pd.DataFrame(comparison_three["results"]).pivot(index=["id", "question"], columns="variant", values="answer"))
else:
    print("Comparação pendente:", comparison_three.get("missing_adapters", comparison_three.get("error")))
update_report(comparison_three, METADATA_02, REPORTS_FASE3_DIR / "relatorio_tecnico_fase3.md")


## 14. Download das evidências
Guarde o ZIP e baixe também este notebook **executado, com as saídas**.
O ZIP contém adapter 02, metadata, snapshots dos splits, resultados, relatório e revisão Git.
Nenhuma evidência anterior é substituída. Para usar o novo adapter na API,
configure FASE3_ADAPTER_PATH apontando para `models/fase3/experimento_02/adapter`.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/fase3_experimento_02_evidencias.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as zipped:
    for folder in [EXPERIMENT_02_DIR, REPORTS_FASE3_DIR]:
        for path in folder.rglob("*"):
            if path.is_file() and "checkpoints" not in path.parts:
                zipped.write(path, path.relative_to(WORKDIR).as_posix())
    zipped.writestr("git_revision.txt", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode())
    zipped.writestr("runtime_dependencies.txt", subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode())
files.download(str(archive))
